# 09 — Blind 100-Image Holdout Evaluation

Holdout is 50 real + 50 fake. No fitting, feature selection, GridSearchCV, or threshold tuning. The frozen Logistic Regression model and frozen threshold are used exactly as saved.

In [1]:
from pathlib import Path
import sys,json,joblib
import pandas as pd
ROOT=Path.cwd()
while ROOT!=ROOT.parent and not (ROOT/"data").exists(): ROOT=ROOT.parent
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from src.features import extract_all
from src.modeling import evaluate
s=json.loads((ROOT/"models/feature_schema_l1.json").read_text()); model=joblib.load(ROOT/"models/model_frozen.joblib",mmap_mode="r")
selected=s["selected_features"]; holdout=ROOT/"data/holdout_50"; assert holdout.exists(),f"Missing holdout: {holdout}"
IMAGE_EXTENSIONS={".jpg",".jpeg",".png",".bmp",".webp",".tif",".tiff"}
rows=[]; labels=[]
for cls,y in [("real",0),("fake",1)]:
    folder=holdout/cls; assert folder.exists(),f"Missing: {folder}"
    paths=sorted(p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)
    print(cls,"image files:",len(paths))
    print("sample:",[p.name for p in paths[:5]])
    assert len(paths)==50,f"Expected 50 images in {folder}, found {len(paths)}"
    for p in paths: rows.append(extract_all(p)); labels.append(y)
X=pd.DataFrame(rows).reindex(columns=selected)
p=model.predict_proba(X)[:,1]
metrics=evaluate(labels,p,float(s["threshold"]))
print("BLIND HOLDOUT METRICS:"); print(json.dumps(metrics,indent=2))
out=ROOT/"metrics/holdout_100_metrics.json"; out.write_text(json.dumps(metrics,indent=2),encoding="utf-8"); print("Saved:",out)


C:\Users\devar\AppData\Local\Programs\Python\Python312\Lib\contextlib.py:137: UserWarning: mmap_mode "r" is not compatible with compressed file d:\deepfake_noise_wavelet_ml\models\model_frozen.joblib. "r" flag will be ignored.
  return next(self.gen)


real image files: 50
sample: ['real_001.jpg', 'real_002.jpg', 'real_003.jpg', 'real_004.jpg', 'real_005.jpg']
fake image files: 50
sample: ['ai_generated_001.jpg', 'ai_generated_002.jpg', 'ai_generated_003.jpg', 'ai_generated_004.jpg', 'ai_generated_005.jpg']
BLIND HOLDOUT METRICS:
{
  "roc_auc": 0.4872,
  "pr_auc": 0.4587853250038666,
  "accuracy": 0.56,
  "precision": 0.5428571428571428,
  "recall": 0.76,
  "f1": 0.6333333333333332,
  "balanced_accuracy": 0.56,
  "tn": 18,
  "fp": 32,
  "fn": 12,
  "tp": 38
}
Saved: d:\deepfake_noise_wavelet_ml\metrics\holdout_100_metrics.json


d:\deepfake_noise_wavelet_ml\.venv\Lib\site-packages\sklearn\base.py:458: UserWarning: X has feature names, but SimpleImputer was fitted without feature names
  warnings.warn(
